In [1]:
%%capture
pip install transformer_lens transformers python-dotenv matplotlib seaborn jaxtyping colorama openai

In [2]:
# Utils
import os, time, re, io, json, requests, random

from datetime import datetime
from dotenv import load_dotenv
from zoneinfo import ZoneInfo
from tqdm import tqdm

# Data Visualisations
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ML
import torch
from torch import Tensor

# Annotations and Types
from jaxtyping import Float, Int
from typing import List, Callable
from colorama import Fore

# Mech Interp.
from transformer_lens.hook_points import HookPoint
from transformer_lens import HookedTransformer, utils
from transformers import AutoTokenizer

# OpenAI - API
from openai import OpenAI
from functools import partial

/workspace/Algoverse_Mech_Interp/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/Algoverse_Mech_Interp/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/workspace/Algoverse_Mech_Interp/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarni

In [3]:
def getDevice():
    if torch.cuda.is_available(): #nvidia/runpod
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps") #apple silicon
    else:
        return torch.device("cpu")
    
DEVICE = getDevice()
print(DEVICE)

cuda


In [ ]:
model = HookedTransformer.from_pretrained("Qwen/Qwen1.5-4B-Chat", device=DEVICE)s

In [17]:
model.cfg.n_layers

40

In [ ]:
# ...existing code...
layer_id = 30

cache_name = f"blocks.{layer_id}.hook_resid_post"
_, cache = model.run_with_cache("Love")
act_love = cache[cache_name]    # (1, seq_len1, d_model)
_, cache = model.run_with_cache("Hate")
act_hate = cache[cache_name]    # (1, seq_len2, d_model)

# make a single steering vector (d_model,) so it can be safely broadcast
steering_vec = (act_love.mean(dim=1) - act_hate.mean(dim=1)).squeeze(0)

def act_add(steering_vec):
    def hook(activation, hook):           # must accept (activation, hook)
        sv = steering_vec.to(activation.device).to(dtype=activation.dtype)
        return activation + sv.view(1, 1, -1)   # broadcast over batch and sequence
    return hook

test_sentence = "I think dogs are "
model.add_hook(name=cache_name, hook=act_add(steering_vec))
print(model.generate(test_sentence, max_new_tokens=10))
model.reset_hooks()

# apply the negative steering vector
model.add_hook(name=cache_name, hook=act_add(-steering_vec))
print(model.generate(test_sentence, max_new_tokens=10))
model.reset_hooks()
# ...existing code...

 40%|████      | 4/10 [00:00<00:00, 30.53it/s]

100%|██████████| 10/10 [00:00<00:00, 30.21it/s]


I think dogs are )((((.Topic比ⓜ});



})();
.CompareTo cant枅{!!


100%|██████████| 10/10 [00:00<00:00, 29.56it/s]

I think dogs are 三天 AutoMapper dout AutoMapper AutoMapper AutoMapper AutoMapper说我滥 AutoMapper
